# UJIIndoorLoc trên Google Colab

Notebook lấy mã nền từ `main`, sau đó tự bổ sung plugin mới từ tất cả nhánh `feature/*` mà token đọc được. Menu phát hiện model và cách xử lý RSS từ các nhánh đã tải, gồm KNN/WKNN, RSS powered, CDAE-CNN và SAE/CDAE-MMJ. File đã có ở `main` không bị ghi đè; thay đổi file dùng chung cần được merge vào `main`. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy model neural nặng. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation.


In [ ]:
import io
import json
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files, userdata

REPO = "VietUniverse/ips-wifi-research"
API = f"https://api.github.com/repos/{REPO}"
root = Path("/content/repo_source")
source_cache = root.parent / "feature_sources"
try:
    token = str(userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    token = ""
if not token:
    raise RuntimeError("Repo private: trong Colab → Secrets, thêm GITHUB_TOKEN có quyền Contents: read và Metadata: read cho VietUniverse/ips-wifi-research, bật quyền truy cập cho notebook rồi chạy lại ô này.")
headers = {"Accept": "application/vnd.github+json", "Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
token = None  # Không giữ token trong biến sau khi tạo headers

def github_get(url, timeout=120):
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.read()
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError("GitHub HTTP 401: token sai/hết hạn hoặc thiếu quyền Contents: read + Metadata: read. Tạo token mới, cập nhật Colab Secret GITHUB_TOKEN, rồi chạy lại ô đầu.") from None
        if error.code == 404:
            raise RuntimeError("GitHub HTTP 404: repo/nhánh không truy cập được. Kiểm tra tên repo và quyền của token với repo private.") from None
        raise RuntimeError(f"GitHub API lỗi HTTP {error.code}; kiểm tra mạng hoặc quyền token.") from None
    except urllib.error.URLError as error:
        raise RuntimeError(f"Không kết nối được GitHub: {error.reason}") from None

def extract_branch(branch, destination):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    archive = github_get(f"{API}/zipball/{branch}")
    with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
        zipped.extractall(destination)
    project_files = list(destination.rglob("pyproject.toml"))
    if len(project_files) != 1:
        raise RuntimeError(f"Nhánh {branch}: cần đúng một pyproject.toml, tìm thấy {len(project_files)}")
    return project_files[0].parent

for folder in (root, source_cache):
    if folder.exists():
        shutil.rmtree(folder)
root.mkdir(parents=True)
source_cache.mkdir(parents=True)
project = extract_branch("main", root)
branch_list = json.loads(github_get(f"{API}/branches?per_page=100", timeout=30))
feature_branches = sorted(item["name"] for item in branch_list if item["name"].startswith("feature/"))
if not feature_branches:
    print("GitHub không trả về nhánh feature/*. Sẽ dùng model đã có trong main.")
print("Mã nền: main")
print("Đang quét các nhánh nhóm:", feature_branches or "(chưa có)")

def is_plugin_or_config(relative):
    parts = relative.parts
    if len(parts) == 4 and parts[:3] == ("src", "ips", "algorithms") and relative.suffix == ".py":
        return True
    if len(parts) == 5 and parts[:4] == ("src", "ips", "data", "preprocessors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "neighbors") and relative.suffix == ".py":
        return True
    if len(parts) == 4 and parts[:3] == ("src", "ips", "experiments") and parts[3].startswith("knn_") and relative.suffix == ".py":
        return True
    if len(parts) == 2 and parts[0] == "configs" and relative.suffix == ".json":
        return True
    return False

loaded, skipped = {}, []
for branch in feature_branches:
    branch_dir = source_cache / branch.replace("/", "__")
    try:
        branch_project = extract_branch(branch, branch_dir)
    except (RuntimeError, zipfile.BadZipFile) as error:
        print(f"Bỏ qua {branch}: {error}")
        continue
    count = 0
    for source in branch_project.rglob("*"):
        if not source.is_file():
            continue
        relative = source.relative_to(branch_project)
        if not is_plugin_or_config(relative):
            continue
        target = project / relative
        if target.exists():
            skipped.append(f"{branch}:{relative.as_posix()}")
            continue  # main/nhánh đã tải trước là nguồn chuẩn cho file trùng tên
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        count += 1
    loaded[branch] = count
    print(f"  {branch}: thêm {count} file plugin/config")

print("Tổng file trùng tên được giữ nguyên theo main/nhánh đầu tiên:", len(skipped))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project)], check=True)
sys.path.insert(0, str(project / "src"))
print("Project:", project, "| Python:", sys.executable)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn model và tiền xử lý

Ô sau in danh sách model, tiền xử lý RSS và preset đã tải được, rồi **dừng để bạn nhập lựa chọn**. Có thể chọn nhiều model cùng họ bằng các số cách nhau dấu phẩy. KNN/WKNN và RSS powered đến từ nhánh KNN; CDAE-CNN và SAE/CDAE-MMJ đến từ các nhánh model tương ứng. Sau khi thành viên push plugin/config mới, chạy lại notebook từ ô đầu để nhận mã mới. Preset Mixer chỉ chạy với `bsrbf_kan` và `rss_standard` (520 AP).


In [ ]:
import json

options = json.loads(subprocess.check_output(
    [sys.executable, "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(str(p.relative_to(project)) for p in (project / "configs").glob("*.json"))
families = {"Neural": options["models"]}
if (project / "src/ips/neighbors/__init__.py").is_file():
    neighbor_probe = subprocess.run(
        [sys.executable, "-c", "import json; from ips.neighbors import available_models; print(json.dumps(sorted(available_models())))"],
        capture_output=True, text=True)
    if neighbor_probe.returncode:
        raise RuntimeError(f"Không tải được plugin KNN: {neighbor_probe.stderr}")
    families["KNN/WKNN"] = json.loads(neighbor_probe.stdout)

def choose_one(title, values):
    if not values: raise ValueError(f'Không có lựa chọn cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input(f'Nhập số 1..{len(values)}: ').strip()
        if raw.isdigit() and 1 <= int(raw) <= len(values): return values[int(raw)-1]
        print('Số không hợp lệ, hãy nhập lại.')

def choose_many(title, values):
    if not values: raise ValueError(f'Không có model cho {title}')
    print('\n' + title)
    for i, value in enumerate(values, 1): print(f'  {i}. {value}')
    while True:
        raw = input('Nhập số model, ví dụ 1 hoặc 1,3: ').strip()
        try:
            indices = [int(x.strip()) for x in raw.split(',')]
            if indices and len(indices) == len(set(indices)) and all(1 <= i <= len(values) for i in indices):
                return [values[i-1] for i in indices]
        except ValueError: pass
        print('Danh sách không hợp lệ, hãy nhập lại.')

selected_family = choose_one('Chọn họ model', list(families))
selected_models = choose_many('Chọn model', families[selected_family])
selected_preprocess = choose_one('Chọn cách xử lý RSS', options['preprocessors'])
family_presets = [p for p in presets if Path(p).name.startswith('knn_') == (selected_family == 'KNN/WKNN')]
selected_preset = choose_one('Chọn preset', family_presets)
preprocess_params = json.loads(input('RSS params dạng JSON (Enter để dùng {}): ').strip() or '{}')
if not isinstance(preprocess_params, dict): raise ValueError('RSS params phải là JSON object')
print('\nĐã chọn:', selected_family, selected_models, selected_preprocess, selected_preset)


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
if not selected_models or not selected_preprocess or not selected_preset:
    raise ValueError("Hãy chọn model, RSS và preset")
settings = json.loads((project / selected_preset).read_text())
print('Chạy:', selected_models, '| Tiền xử lý:', selected_preprocess, '| Preset:', selected_preset)
print('Cấu hình preset:', json.dumps(settings, ensure_ascii=False, indent=None if selected_family == "KNN/WKNN" else 2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
from ips.experiments.visualize import render_run, compare_runs
run_dirs = []
if selected_family == "KNN/WKNN":
    settings.update(models=selected_models, preprocess=selected_preprocess,
                    preprocess_params=preprocess_params)
    selected_config = Path("/content") / f"selected_{RUN_NAME}.json"
    selected_config.write_text(json.dumps(settings), encoding="utf-8")
    knn_run = subprocess.run([sys.executable, "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                              "--data-dir", str(data_dir), "--output-dir", str(run_dir)],
                             capture_output=True, text=True)
    if knn_run.returncode:
        print(knn_run.stdout)
        print(knn_run.stderr)
        raise RuntimeError("knn_cli lỗi - xem thông báo ở trên")
    (run_dir / "console_log.txt").write_text(knn_run.stdout, encoding="utf-8")  # log đầy đủ nằm trong ZIP kết quả
    print("\n".join(line for line in knn_run.stdout.splitlines() if line.startswith("[tòa")))
    knn_table, knn_figures = render_run(run_dir, show=False)  # bảng/ảnh của repo: xem ở Phụ lục bên dưới
    run_dirs.append(run_dir)
else:
    if settings.get("with_mixer") and (selected_models != ["bsrbf_kan"] or selected_preprocess != "rss_standard"):
        raise ValueError("Preset Mixer cần model bsrbf_kan và RSS rss_standard")
    print("GPU available:", torch.cuda.is_available())
    if "bsrbf_kan" in selected_models and not torch.cuda.is_available():
        print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
    for selected_model in selected_models:
        model_dir = run_dir / selected_model
        print('Đang chạy:', selected_model)
        subprocess.run([sys.executable, "-u", "-m", "ips.experiments.cli", "--config", str(project / selected_preset),
                        "--model", selected_model, "--preprocess", selected_preprocess,
                        "--preprocess-params", json.dumps(preprocess_params),
                        "--data-dir", str(data_dir), "--output-dir", str(model_dir)], check=True)
        render_run(model_dir)
        run_dirs.append(model_dir)
    if len(run_dirs) > 1:
        compare_runs(run_dirs, labels=selected_models)
print('Kết quả:', run_dir)


## Báo cáo họ KNN/WKNN (hiện khi chọn họ KNN/WKNN ở ô chọn model)

Các ô bên dưới trình bày kết quả như notebook Lần 4: **Phần B** bảng, **Phần C** biểu đồ, **Phần D** so sánh + kiểm định + kết luận, rồi Phụ lục. Nếu bạn chạy họ Neural, các ô này chỉ báo "không áp dụng" và không ảnh hưởng gì.

**Cách đọc các chỉ số** (mỗi mẫu test là một lần đo Wi-Fi tại vị trí đã biết thật; mô hình đoán tầng và tọa độ, tòa nhà đã biết trước):

| Chỉ số | Ý nghĩa |
|---|---|
| Floor accuracy (%) | Tỉ lệ mẫu đoán đúng tầng. Vote tầng giống hệt nhau nên **bằng nhau** giữa ba phương pháp ở cùng k. |
| Mean / Median / P90 (m) | Sai số vị trí trung bình / trung vị / phân vị 90 trên **tất cả** mẫu test. Nhỏ hơn là tốt hơn. |
| ≤1 m, ≤3 m, ≤5 m (%) | Tỉ lệ mẫu có sai số trong ngưỡng đó. Lớn hơn là tốt hơn. |
| Validation mean (m) | Sai số trên validation nội bộ, dùng **để chọn k** (official test chỉ để báo cáo). |
| Thời gian | `Gộp tọa độ` là phần duy nhất các phương pháp khác nhau; `Tổng` gồm cả bước tìm láng giềng dùng chung. |

Giao thức này (theo repo) khác Colab Lần 4 (tham chiếu 100% train, tự vote tòa, chọn k trên test) nên số không so trực tiếp được.

In [ ]:
import pandas as pd
from contextlib import contextmanager
from IPython.display import Image, Markdown, display

knn_ready = selected_family == "KNN/WKNN" and (run_dir / "sweep_by_k.csv").exists()


@contextmanager
def guarded(title):
    try:
        yield
    except Exception as error:
        print(f"[{title}] không hiển thị được: {error}")


def read_table(name, **kwargs):
    path = run_dir / name
    try:
        return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()
    except pd.errors.EmptyDataError:
        return pd.DataFrame()  # ví dụ chỉ chạy một model thì không có bảng so sánh cặp


def show_chart(name):
    path = run_dir / name
    if path.exists():
        display(Image(filename=str(path), width=820))
    else:
        print(f"Chưa có {name} (có thể thiếu matplotlib khi chạy).")


if not knn_ready:
    print("Báo cáo này chỉ dành cho họ KNN/WKNN (chọn họ KNN/WKNN ở ô chọn model rồi chạy ô trên).")
else:
    from ips.neighbors import build_model
    sweep = read_table("sweep_by_k.csv")
    MODELS = [m for m in selected_models if m in set(sweep.method)]
    NAME = {m: build_model(m).display_name for m in MODELS}
    summary_table = read_table("summary_table.csv")
    d1 = read_table("pairwise_same_k.csv", index_col=0)
    d2 = read_table("wilcoxon.csv")
    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    selected_k = {m: int(sweep[sweep.method == m].sort_values(["val_mean", "k"]).iloc[0].k) for m in MODELS}
    n_test = next(iter(summary.get("official_test", {}).values()), {}).get("n", "?")
    log_lines = (run_dir / "console_log.txt").read_text(encoding="utf-8").splitlines()
    start = next((i for i, line in enumerate(log_lines) if line.startswith("=" * 40)), None)
    end = next((i for i, line in enumerate(log_lines) if line.startswith("Lưu ý")), None)
    conclusion = log_lines[start:end + 1] if start is not None and end is not None else []
    print(f"Đã chạy xong: {len(MODELS)} model × {sweep.k.nunique()} giá trị k = {len(sweep)} lần đánh giá trên {n_test} mẫu test")
    print("k chọn trên validation (sai số trung bình thấp nhất): " + " | ".join(f"{NAME[m]} k={selected_k[m]}" for m in MODELS))

In [ ]:
if not knn_ready:
    print("Chỉ dành cho họ KNN/WKNN.")
else:
    display(Markdown("# PHẦN B — Kết quả dạng bảng"))
    with guarded("B1"):
        display(Markdown("### B1. Độ chính xác tầng (%) theo k\nCác cột giống nhau là bình thường (xem cách đọc ở trên)."))
        display(sweep.pivot(index="k", columns="method", values="floor_acc")[MODELS].rename(columns=NAME).round(2))
    with guarded("B2"):
        display(Markdown("### B2. Sai số vị trí theo k (mét) — official test\nMean, Median và P90; số nhỏ hơn là tốt hơn."))
        metric_label = {"test_mean": "Mean", "test_median": "Median", "test_p90": "P90"}
        t_err = sweep.pivot(index="k", columns="method", values=list(metric_label))
        t_err.columns = pd.MultiIndex.from_tuples([(metric_label[metric], NAME[m]) for metric, m in t_err.columns])
        display(t_err[[(metric_label[c], NAME[m]) for c in metric_label for m in MODELS]].round(3))
    with guarded("B2b"):
        display(Markdown("### B2b. Sai số trung bình theo k (mét) — validation nội bộ\nBảng dùng **để chọn k** (k có sai số nhỏ nhất, hòa thì chọn k nhỏ hơn)."))
        display(sweep.pivot(index="k", columns="method", values="val_mean")[MODELS].rename(columns=NAME).round(3))
        print("k chọn:", " | ".join(f"{NAME[m]} k={selected_k[m]}" for m in MODELS))
    with guarded("B3"):
        display(Markdown("### B3. Thời gian theo k (giây, cho toàn bộ tập test)\n`Gộp tọa độ` là phần duy nhất các phương pháp khác nhau; `Tổng` gồm cả bước tìm láng giềng dùng chung."))
        time_label = {"t_predict_s": "Gộp tọa độ", "total_s": "Tổng"}
        t_time = sweep.pivot(index="k", columns="method", values=list(time_label))
        t_time.columns = pd.MultiIndex.from_tuples([(time_label[metric], NAME[m]) for metric, m in t_time.columns])
        display(t_time[[(time_label[c], NAME[m]) for c in time_label for m in MODELS]].round(4))
        print(f"Thời gian tìm láng giềng (dùng chung, đo một lần): {summary.get('t_search_test_s', float('nan')):.2f} s cho {n_test} mẫu test")
    with guarded("B4"):
        display(Markdown("### B4. Bảng tổng hợp tại các k quan trọng\nGồm k chọn trên validation của từng phương pháp, cả nhóm ở cùng một k (để so công bằng) và k tham chiếu của preset."))
        display(summary_table.round(3))

In [ ]:
CHARTS = [
    ("C1. Độ chính xác tầng theo k", "Trục ngang là k, trục dọc là % mẫu đoán đúng tầng. Các đường **trùng nhau** vì vote tầng giống hệt nhau (không phải lỗi).", "C1_floor_accuracy.png"),
    ("C2. Sai số vị trí trung bình theo k (official test)", "Đường **thấp hơn = chính xác hơn**. Đường đứt nét đánh dấu k chọn trên validation. Ở k = 1 các đường trùng nhau (chỉ có một láng giềng nên trọng số không còn ý nghĩa).", "C2_test_mean_error.png"),
    ("C3. Sai số trung bình theo k (validation nội bộ)", "Như C2 nhưng trên validation, là đường dùng để chọn k. Nếu C2 và C3 cùng xu hướng thì việc chọn k trên validation đáng tin hơn.", "C3_val_mean_error.png"),
    ("C4. Phân bố sai số (CDF) tại k chọn của mỗi phương pháp", "Tại điểm x trên trục ngang, chiều cao đường là tỉ lệ mẫu có sai số ≤ x mét. **Đường cao hơn / lệch trái hơn = tốt hơn.**", "C4_cdf.png"),
    ("C5. Thời gian chạy theo k", "Đường thấp hơn = nhanh hơn. Thời gian tổng gồm bước tìm láng giềng (chiếm gần hết, giống nhau ở mọi phương pháp) nên các đường thường rất sát nhau.", "C5_time.png"),
    ("C6. Chênh lệch sai số giữa các cặp phương pháp ở từng k", "Mỗi cột là `sai số phương pháp A − sai số phương pháp B` ở một giá trị k. **Cột dương = B chính xác hơn A**, cột âm = B kém hơn.", "C6_chenh_lech.png"),
]
if not knn_ready:
    print("Chỉ dành cho họ KNN/WKNN.")
else:
    display(Markdown("# PHẦN C — Biểu đồ\nMàu **cam = kNN**, **xanh dương = WKNN (1/d)**, **xanh lá = WKNN (1/d²)**."))
    for title, guide, png in CHARTS:
        with guarded(title):
            display(Markdown(f"### {title}\n**Cách đọc:** {guide}"))
            show_chart(png)

In [ ]:
if not knn_ready:
    print("Chỉ dành cho họ KNN/WKNN.")
else:
    display(Markdown("# PHẦN D — Đánh giá: các phương pháp khác nhau thế nào?"))
    with guarded("D1"):
        display(Markdown("### D1. So sánh ở cùng k (k = 1 → k_max)\nHiệu số sai số trung bình (mét) của từng cặp ở mỗi k, **không phụ thuộc** việc chọn k. **Dương = phương pháp thứ hai trong cặp chính xác hơn.**"))
        if d1.empty:
            print("Chỉ có một model nên không có phần so sánh cặp.")
        else:
            display(d1.round(4))
            for col in d1.columns:
                v = d1[col].iloc[1:]
                print(f"{col} (k = 2..{d1.index.max()}): {col.split(' − ')[1]} tốt hơn ở {int((v > 1e-9).sum())}/{len(v)} giá trị k, "
                      f"kém hơn ở {int((v < -1e-9).sum())}/{len(v)}; chênh trung bình {v.mean():+.3f} m")
    with guarded("D2"):
        display(Markdown("### D2. So sánh từng mẫu + kiểm định thống kê (Wilcoxon cặp)\nVới cùng một mẫu test, xem phương pháp nào có sai số nhỏ hơn, rồi dùng **Wilcoxon cặp** để biết chênh lệch có ý nghĩa thống kê hay chỉ do ngẫu nhiên (ngưỡng Bonferroni = 0,05 / số cặp). **Có ý nghĩa thống kê chưa chắc có ý nghĩa thực tế**: hãy xem cả cột chênh lệch (m)."))
        if d2.empty:
            print("Chỉ có một model nên không có kiểm định cặp.")
        else:
            alpha = 0.05 / d2.groupby("k").size().iloc[0]
            cols = {"cap": "Cặp (A − B)", "mean_a": "Mean A (m)", "mean_b": "Mean B (m)", "diff_a_minus_b": "A − B (m)",
                    "pct_b_better": "% mẫu B nhỏ hơn", "pct_a_better": "% mẫu A nhỏ hơn", "pct_tie": "% bằng nhau",
                    "p_value": "p (Wilcoxon)", "significant": "Có ý nghĩa"}
            for k_cmp, group in d2.groupby("k"):
                print(f"\n=== Tại k = {k_cmp} ({n_test} mẫu test; ngưỡng Bonferroni {alpha:.4f}) ===")
                shown = group[list(cols)].rename(columns=cols).reset_index(drop=True)
                display(shown.style.format({"Mean A (m)": "{:.3f}", "Mean B (m)": "{:.3f}", "A − B (m)": "{:+.3f}",
                                            "% mẫu B nhỏ hơn": "{:.1f}", "% mẫu A nhỏ hơn": "{:.1f}", "% bằng nhau": "{:.1f}",
                                            "p (Wilcoxon)": "{:.3g}"}).hide(axis="index"))
    with guarded("D3"):
        display(Markdown("### D3. Kết luận tự động\nTóm tắt ba tiêu chí: độ chính xác tầng, sai số vị trí (k chọn trên validation và cùng k), thời gian."))
        print("\n".join(conclusion) if conclusion else "Không tìm thấy khung kết luận; xem console_log.txt trong thư mục kết quả.")

In [ ]:
if not knn_ready:
    print("Chỉ dành cho họ KNN/WKNN.")
else:
    with guarded("Phụ lục"):
        display(Markdown("# PHỤ LỤC — Bảng và biểu đồ bổ sung của repo\nSai số theo từng tòa, CDF và ma trận nhầm lẫn **tầng** (do `render_run` của repo vẽ; ảnh nằm trong thư mục `figures` của kết quả)."))
        display(knn_table.round(3))
        for path in knn_figures:
            display(Image(filename=str(path)))

In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
